**Подготовка файла 2800.csv на основе данных сводных файлов 2016-2018 годов**

- Автор: Клинкова Екатерина
- Telegram: @ekaklinkova

- `..\Data\{год}\` - папка для хранения созданнного 2200.csv для определенного года
- `..\Data\{год}\Original` - папка для хранения оригинального файла xlsx
- `..\Data\Common\` - папка для хранения итогового результата 2200.csv сводный

Пример: 
- *E:/IrkutskProject/Data/2016/Original/ф33  правл за 2016г Иркутская обл.xlsx*
- *E:/IrkutskProject/Data/2016/2200.csv*

In [4]:
# Загрузка библиотек
import pandas as pd
import numpy as np

In [5]:
# Установка библиотеки для чтения excel. 
# !pip install openpyxl
# !pip install xlrd

In [6]:
# Проверка версии
# pip show openpyxl

1) Подкотовка датасетов формы 2800 за 2016-2018 года

In [8]:
# Список годов для обработки
years = [2016, 2017, 2018] #, 2017, 2018]

# Название оригинального файла
original_names = {
    2016: 'ф33  правл за 2016г Иркутская обл.xlsx',
    2017: 'ф.33 и ф.8 Иркутская область 2017г..xls',
    2018: 'ф.33 и ф.8  Иркутская область  2018.xls'
}
original_url = {}
path = 'E:/IrkutskProject/Data/'

df_2800 = {}

for year in years:
    original_path = f'{path}{year}/Original/'
    original_url[year] = original_path + original_names[year]
    
    if year == 2016:
        df = pd.read_excel(original_url[year], engine='openpyxl', sheet_name = 6)

        # Отфильтруем таблицу 2800
        df_2800[year] = df.iloc[27:36]

        # Удаляем пустые столбцы
        df_2800[year] = df_2800[year].drop(df_2800[year].columns[114:164], axis=1)
        df_2800[year] = df_2800[year].drop(df_2800[year].columns[87:113], axis=1)
        df_2800[year] = df_2800[year].drop(df_2800[year].columns[60:86], axis=1)
        df_2800[year] = df_2800[year].drop(df_2800[year].columns[1:59], axis=1)

        # Удаляем лишние строки
        df_2800[year].drop(df_2800[year].index[0:2], inplace=True)
    else:
        if year == 2017:
            df = pd.read_excel(original_url[year], engine='xlrd', sheet_name=8)
            
        elif year == 2018:
            df = pd.read_excel(original_url[year], engine='xlrd', sheet_name=4)
            
        # Отфильтруем таблицу 2800
        df_2800[year] = df.iloc[26:33]
        df_2800[year] = df_2800[year].drop(df_2800[year].columns[18:19], axis=1)
        df_2800[year] = df_2800[year].drop(df_2800[year].columns[12:15], axis=1)
        df_2800[year] = df_2800[year].drop(df_2800[year].columns[0:11], axis=1)

    # Переименовываем колонки
    df_2800[year].columns = ['Наименование показателей', 'Всего', 'Из них детей от 0 до 14 лет', 'Из них подростков от 15 до 17 лет']
    
    # Подписываем строки в соответствии с шаблоном
    df_2800[year].iloc[0, 0] = 'Выявлено в текущем году'
    df_2800[year].iloc[1, 0] = 'Выявлено в текущем году, из  них с МБТ+'
    df_2800[year].iloc[2, 0] = 'Наблюдалось в отчетном году'
    df_2800[year].iloc[3, 0] = 'Лечились в стационаре'
    df_2800[year].iloc[4, 0] = 'Лечились амбулаторно'
    df_2800[year].iloc[5, 0] = 'Умерло от туберкулеза всего'
    df_2800[year].iloc[6, 0] = 'Умерло от туберкулеза всего, из них в стационаре'
    
    df_2800[year] = df_2800[year].reset_index(drop=True)
    df_2800[year].to_csv(f'E:/IrkutskProject/Data/{year}/2800.csv')

In [9]:
df_2800[2018]

,Наименование показателей,Всего,Из них детей от 0 до 14 лет,Из них подростков от 15 до 17 лет
0,Выявлено в текущем году,29,NaN,1
1,"Выявлено в текущем году, из них с МБТ+",2,NaN,1
2,Наблюдалось в отчетном году,4,NaN,NaN
3,Лечились в стационаре,2,NaN,NaN
4,Лечились амбулаторно,2,NaN,NaN
5,Умерло от туберкулеза всего,NaN,NaN,NaN
6,"Умерло от туберкулеза всего, из них в стационаре",NaN,NaN,NaN


In [10]:
columns = [
    'Показатель',
    'Возраст',
    'Год',
    'Значение'
]

data = []

df_2800_long = pd.DataFrame(columns=columns)

for year in years:
    for row_number in range(df_2800[year].shape[0]):
        for col_number in range(1, 4):
            new_row = []
            new_row.append(df_2800[year].iloc[row_number, 0].strip())
            new_row.append(df_2800[year].columns[col_number])
            new_row.append(year)
            new_row.append(df_2800[year].iloc[row_number, col_number] if pd.notnull(df_2800[year].iloc[row_number, col_number]) else 0)
            data.append(new_row)
df_2800_long = pd.DataFrame(data, columns=columns)

df_2800_long

,Показатель,Возраст,Год,Значение
0,Выявлено в текущем году,Всего,2016,16
1,Выявлено в текущем году,Из них детей от 0 до 14 лет,2016,0
2,Выявлено в текущем году,Из них подростков от 15 до 17 лет,2016,0
3,"Выявлено в текущем году, из них с МБТ+",Всего,2016,1
4,"Выявлено в текущем году, из них с МБТ+",Из них детей от 0 до 14 лет,2016,0
...,...,...,...,...
58,Умерло от туберкулеза всего,Из них детей от 0 до 14 лет,2018,0
59,Умерло от туберкулеза всего,Из них подростков от 15 до 17 лет,2018,0
60,"Умерло от туберкулеза всего, из них в стационаре",Всего,2018,0
61,"Умерло от туберкулеза всего, из них в стационаре",Из них детей от 0 до 14 лет,2018,0


In [17]:
df_2800_long.to_csv(f'{path}Common/2800.csv', index=False)